# 🔄 Master Orchestration Workflow
## Real-Time Sales Performance & Inventory Analytics Platform

**Purpose**: Orchestrate the complete data pipeline from Bronze → Silver → Gold

**Storage**: AWS S3 External Storage (Unity Catalog External Schemas)
- 🥉 Bronze: `workspace.`workspace-adventureworks-bronze`` (S3)
- 🥈 Silver: `workspace.`workspace-adventureworks-silver`` (S3)
- 🥇 Gold: `workspace.`workspace-adventureworks-gold`` (S3)

**Execution Order**:
1. ✅ **Setup**: Verify schemas exist (external storage on S3)
2. 🥉 **Bronze Layer**: Ingest all source data to S3
3. 🥈 **Silver Layer**: Build dimensions and facts on S3
4. 🥇 **Gold Layer**: Create business aggregates on S3

**Features**:
- ✅ Automated dependency management
- ✅ Sequential execution for serverless single-node
- ✅ Error handling and retry logic
- ✅ Execution time tracking
- ✅ Comprehensive logging
- ✅ S3 external storage for data persistence

**Can be scheduled as**: Databricks Job (Daily/Hourly)

---

In [0]:
from datetime import datetime
import time

# Workflow Configuration for Serverless Compute
MAX_WORKERS = 1  # Sequential execution for single-node serverless
TIMEOUT_SECONDS = 3600  # 1 hour timeout per notebook
RETRY_ATTEMPTS = 2  # Number of retry attempts for failed notebooks

# Base path for notebooks (where code lives)
BASE_PATH = "/Real-Time-Sales-Performance-Inventory-Analytics-Platform/Real-Time-Sales-Performance-Inventory-Analytics-Platform"

# External Storage Schemas (AWS S3 backed)
# Note: Schema names contain hyphens, require backticks in SQL
BRONZE_SCHEMA = "workspace.`workspace-adventureworks-bronze`"
SILVER_SCHEMA = "workspace.`workspace-adventureworks-silver`"
GOLD_SCHEMA = "workspace.`workspace-adventureworks-gold`"

# Execution tracking
execution_results = []
start_time = datetime.now()

print("=" * 80)
print("🚀 ADVENTUREWORKS ANALYTICS PIPELINE - ORCHESTRATION STARTED")
print("=" * 80)
print(f"📅 Start Time: {start_time}")
print(f"💻 Compute: Databricks Serverless (Single Node)")
print(f"☁️  Storage: AWS S3 (External Unity Catalog Schemas)")
print(f"🔧 Execution Mode: Sequential (MAX_WORKERS={MAX_WORKERS})")
print(f"⏱️  Timeout per Notebook: {TIMEOUT_SECONDS}s")
print(f"🔄 Retry Attempts: {RETRY_ATTEMPTS}")
print("=" * 80)
print(f"\n📊 Target Schemas:")
print(f"   🥉 Bronze: {BRONZE_SCHEMA}")
print(f"   🥈 Silver: {SILVER_SCHEMA}")
print(f"   🥇 Gold: {GOLD_SCHEMA}")
print("=" * 80)
print()
print("ℹ️  Serverless compute will auto-start for each notebook execution")
print("ℹ️  Cold start may add 1-2 minutes per notebook")
print("ℹ️  Data will be written to AWS S3 via external schemas")
print("=" * 80)
print()

In [0]:
def run_notebook_with_retry(notebook_path, notebook_name, timeout=TIMEOUT_SECONDS, max_retries=RETRY_ATTEMPTS):
    """
    Run a notebook with retry logic and error handling
    Optimized for Databricks Serverless Compute
    
    Args:
        notebook_path: Relative path to the notebook
        notebook_name: Display name for logging
        timeout: Timeout in seconds
        max_retries: Number of retry attempts
    
    Returns:
        dict: Execution result with status, duration, and error info
    """
    result = {
        "notebook": notebook_name,
        "path": notebook_path,
        "status": "failed",
        "duration_seconds": 0,
        "error": None,
        "attempts": 0
    }
    
    for attempt in range(1, max_retries + 1):
        result["attempts"] = attempt
        start = time.time()
        
        try:
            print(f"   ⏳ Attempt {attempt}/{max_retries}: Running {notebook_name}...")
            if attempt == 1:
                print(f"      (Serverless compute starting - may take 1-2 min for cold start)")
            
            # Run the notebook on serverless compute
            dbutils.notebook.run(notebook_path, timeout)
            
            duration = time.time() - start
            result["status"] = "success"
            result["duration_seconds"] = round(duration, 2)
            
            print(f"   ✅ SUCCESS: {notebook_name} completed in {duration:.2f}s")
            return result
            
        except Exception as e:
            duration = time.time() - start
            result["duration_seconds"] = round(duration, 2)
            result["error"] = str(e)
            
            if attempt < max_retries:
                print(f"   ⚠️  RETRY: {notebook_name} failed (attempt {attempt}). Retrying...")
                print(f"   Error: {str(e)}")
                time.sleep(10)  # Wait 10 seconds before retry (serverless cooldown)
            else:
                print(f"   ❌ FAILED: {notebook_name} failed after {max_retries} attempts")
                print(f"   Error: {str(e)}")
    
    return result


def run_notebooks_sequential(notebooks, stage_name):
    """
    Run multiple notebooks sequentially on Serverless Compute
    Each notebook runs on its own serverless session
    
    Args:
        notebooks: List of tuples (notebook_path, notebook_name)
        stage_name: Name of the execution stage for logging
    
    Returns:
        list: Results from all notebook executions
    """
    print(f"\n{'=' * 80}")
    print(f"🔄 STAGE: {stage_name}")
    print(f"📊 Running {len(notebooks)} notebooks sequentially...")
    print(f"{'=' * 80}")
    
    results = []
    
    for idx, (path, name) in enumerate(notebooks, 1):
        print(f"\n📌 [{idx}/{len(notebooks)}] Starting: {name}")
        result = run_notebook_with_retry(path, name)
        results.append(result)
        
        # Stop if a critical notebook fails
        if result["status"] == "failed":
            print(f"\n⛔ CRITICAL FAILURE: Stopping sequential execution due to failure in {name}")
            break
    
    # Summary
    success_count = sum(1 for r in results if r["status"] == "success")
    failed_count = len(results) - success_count
    total_duration = sum(r["duration_seconds"] for r in results)
    
    print(f"\n{'=' * 80}")
    print(f"📈 {stage_name} - SUMMARY")
    print(f"   ✅ Successful: {success_count}/{len(results)}")
    print(f"   ❌ Failed: {failed_count}/{len(results)}")
    print(f"   ⏱️  Total Duration: {total_duration:.2f}s ({total_duration/60:.2f} min)")
    print(f"{'=' * 80}\n")
    
    return results

In [0]:
# Verify that external S3 schemas exist before starting pipeline
print("\n" + "=" * 80)
print("🔍 VERIFYING EXTERNAL SCHEMAS (AWS S3)")
print("=" * 80)

def verify_schema_exists(schema_full_name, schema_display_name):
    """
    Verify that an external schema exists and is accessible
    """
    try:
        # Extract catalog and schema name "workspace.`workspace-adventureworks-bronze`"
        parts = schema_full_name.replace("`", "").split(".")
        catalog = parts[0]
        schema = parts[1]
        
        # Check if schema exists
        schemas = spark.sql(f"SHOW SCHEMAS IN {catalog}").collect()
        schema_names = [row.databaseName for row in schemas]
        
        if schema in schema_names:
            print(f"   ✅ {schema_display_name}: {schema_full_name} - EXISTS")
            return True
        else:
            print(f"   ❌ {schema_display_name}: {schema_full_name} - NOT FOUND")
            return False
            
    except Exception as e:
        print(f"   ❌ {schema_display_name}: {schema_full_name} - ERROR")
        print(f"      Error: {str(e)}")
        return False

# Verify all three schemas
schemas_to_verify = [
    (BRONZE_SCHEMA, "Bronze Schema"),
    (SILVER_SCHEMA, "Silver Schema"),
    (GOLD_SCHEMA, "Gold Schema")
]

all_schemas_exist = True
for schema_name, display_name in schemas_to_verify:
    if not verify_schema_exists(schema_name, display_name):
        all_schemas_exist = False

print("=" * 80)

if not all_schemas_exist:
    print("\n⛔ PIPELINE ABORTED: One or more external schemas do not exist.")
    print("\n🛠️  Required Actions:")
    print("   1. Create external schemas with S3 storage locations")
    print("   2. Grant appropriate permissions to Unity Catalog")
    print("   3. Verify S3 bucket access from Databricks")
    print("\n📝 Example Schema Creation:")
    print("   CREATE SCHEMA workspace.`workspace-adventureworks-bronze`")
    print("   LOCATION 's3://your-bucket/adventureworks/bronze'")
    print("   COMMENT 'Bronze layer - Raw data on S3';")
    dbutils.notebook.exit("FAILED: External schemas not found")
else:
    print("\n✅ All external schemas verified successfully!")
    print("✅ Ready to proceed with pipeline execution.\n")

In [0]:
# STEP 1: Create Unity Catalog Schemas
print("\n" + "=" * 80)
print("📋 STEP 1: SCHEMA SETUP")
print("=" * 80)

setup_notebooks = [
    (f"{BASE_PATH}/01_Setup_Schemas", "01_Setup_Schemas")
]

setup_results = run_notebooks_sequential(setup_notebooks, "Schema Setup")
execution_results.extend(setup_results)

# Check if setup succeeded
if setup_results[0]["status"] == "failed":
    print("\n⛔ PIPELINE ABORTED: Schema setup failed. Cannot proceed.")
    dbutils.notebook.exit("FAILED: Schema setup failed")

In [0]:
# STEP 2: Bronze Layer - Ingest all source data sequentially
# Sequential execution optimized for single-node serverless compute
bronze_notebooks = [
    (f"{BASE_PATH}/Bronze_layer/02_Bronze_Sales_Ingestion", "Bronze_Sales"),
    (f"{BASE_PATH}/Bronze_layer/02_Bronze_Production_Ingestion", "Bronze_Production"),
    (f"{BASE_PATH}/Bronze_layer/02_Bronze_Purchasing_Ingestion", "Bronze_Purchasing"),
    (f"{BASE_PATH}/Bronze_layer/02_Bronze_Person_Ingestion", "Bronze_Person"),
    (f"{BASE_PATH}/Bronze_layer/02_Bronze_HumanResources_Ingestion", "Bronze_HumanResources")
]

print("\nℹ️  Bronze Layer: Running sequentially on serverless compute")
print("   Each notebook will start its own serverless session")
print("   Expected total time: 10-15 minutes (including cold starts)\n")

bronze_results = run_notebooks_sequential(bronze_notebooks, "🥉 BRONZE LAYER INGESTION")
execution_results.extend(bronze_results)

# Check Bronze layer results
bronze_failed = [r for r in bronze_results if r["status"] == "failed"]
if bronze_failed:
    print(f"\n⚠️  WARNING: {len(bronze_failed)} Bronze notebooks failed:")
    for r in bronze_failed:
        print(f"   - {r['notebook']}: {r['error']}")
    print("\n⚠️  Continuing to Silver layer (with incomplete Bronze data)...")
else:
    print("\n✅ All Bronze notebooks completed successfully!")

In [0]:
# STEP 3: Silver Layer - Build dimensions and facts (sequential for dependencies)
silver_notebooks = [
    (f"{BASE_PATH}/Silver_layer/03_Silver_Dimensions_and_Facts", "Silver_Dimensions_and_Facts"),
]

silver_results = run_notebooks_sequential(silver_notebooks, "🥈 SILVER LAYER TRANSFORMATION")
execution_results.extend(silver_results)

# Check Silver layer results
silver_failed = [r for r in silver_results if r["status"] == "failed"]
if silver_failed:
    print(f"\n⛔ CRITICAL: {len(silver_failed)} Silver notebooks failed:")
    for r in silver_failed:
        print(f"   - {r['notebook']}: {r['error']}")
    print("\n⛔ PIPELINE ABORTED: Cannot proceed to Gold layer without Silver tables.")
    dbutils.notebook.exit("FAILED: Silver layer failed")
else:
    print("\n✅ All Silver notebooks completed successfully!")

In [0]:
# STEP 4: Gold Layer - Create business aggregates sequentially
# Sequential execution for consistent resource usage on serverless
gold_notebooks = [
    (f"{BASE_PATH}/Gold_layer/04_Gold_Sales_Analytics", "Gold_Sales_Analytics"),
    (f"{BASE_PATH}/Gold_layer/04_Gold_Inventory_Analytics", "Gold_Inventory_Analytics"),
    (f"{BASE_PATH}/Gold_layer/04_Gold_Customer_360", "Gold_Customer_360")
]

print("\nℹ️  Gold Layer: Running sequentially on serverless compute")
print("   Expected time: 5-8 minutes\n")

gold_results = run_notebooks_sequential(gold_notebooks, "🥇 GOLD LAYER AGGREGATION")
execution_results.extend(gold_results)

# Check Gold layer results
gold_failed = [r for r in gold_results if r["status"] == "failed"]
if gold_failed:
    print(f"\n⚠️  WARNING: {len(gold_failed)} Gold notebooks failed:")
    for r in gold_failed:
        print(f"   - {r['notebook']}: {r['error']}")
    print("\n⚠️  Pipeline completed with some Gold tables missing.")
else:
    print("\n✅ All Gold notebooks completed successfully!")

In [0]:
# Final Execution Summary
end_time = datetime.now()
total_duration = (end_time - start_time).total_seconds()

print("\n" + "=" * 80)
print("🎉 PIPELINE EXECUTION COMPLETE")
print("=" * 80)
print(f"📅 Start Time: {start_time}")
print(f"📅 End Time: {end_time}")
print(f"⏱️  Total Duration: {total_duration:.2f}s ({total_duration/60:.2f} minutes)")
print("=" * 80)

# Detailed Results by Layer
print("\n📊 DETAILED EXECUTION RESULTS:\n")

layers = {
    "Setup": [r for r in execution_results if "Setup" in r["notebook"]],
    "Bronze": [r for r in execution_results if "Bronze" in r["notebook"]],
    "Silver": [r for r in execution_results if "Silver" in r["notebook"]],
    "Gold": [r for r in execution_results if "Gold" in r["notebook"]]
}

for layer_name, results in layers.items():
    if not results:
        continue
    
    print(f"\n{layer_name} Layer:")
    print("-" * 80)
    
    for r in results:
        status_icon = "✅" if r["status"] == "success" else "❌"
        print(f"  {status_icon} {r['notebook']:<40} | {r['duration_seconds']:>6.2f}s | Attempts: {r['attempts']}")
        if r["error"]:
            print(f"     Error: {r['error'][:100]}...")
    
    success = sum(1 for r in results if r["status"] == "success")
    print(f"\n  Summary: {success}/{len(results)} successful")

# Overall Statistics
total_notebooks = len(execution_results)
successes = sum(1 for r in execution_results if r["status"] == "success")
failures = total_notebooks - successes
success_rate = (successes / total_notebooks * 100) if total_notebooks > 0 else 0

print("\n" + "=" * 80)
print("📈 OVERALL STATISTICS")
print("=" * 80)
print(f"Total Notebooks Executed: {total_notebooks}")
print(f"✅ Successful: {successes}")
print(f"❌ Failed: {failures}")
print(f"📊 Success Rate: {success_rate:.1f}%")
print(f"⏱️  Total Pipeline Duration: {total_duration:.2f}s ({total_duration/60:.2f} min)")
print("=" * 80)

# Determine final status
if failures == 0:
    final_status = "✅ SUCCESS: All notebooks completed successfully!"
    exit_status = "SUCCESS"
elif failures <= 2:
    final_status = f"⚠️  PARTIAL SUCCESS: {failures} notebook(s) failed but pipeline completed"
    exit_status = "PARTIAL_SUCCESS"
else:
    final_status = f"❌ FAILED: {failures} notebook(s) failed"
    exit_status = "FAILED"

print(f"\n{final_status}\n")
print("=" * 80)

# Return status for job monitoring
dbutils.notebook.exit(exit_status)

## 📖 Usage Instructions

### Prerequisites: AWS S3 External Storage Setup

Before running this pipeline, ensure external schemas are created:

```sql
-- Bronze Schema (S3 External Storage)
CREATE SCHEMA IF NOT EXISTS workspace.`workspace-adventureworks-bronze`
LOCATION 's3://your-bucket-name/adventureworks/bronze'
COMMENT 'Bronze layer - Raw data with audit columns on S3';

-- Silver Schema (S3 External Storage)
CREATE SCHEMA IF NOT EXISTS workspace.`workspace-adventureworks-silver`
LOCATION 's3://your-bucket-name/adventureworks/silver'
COMMENT 'Silver layer - Cleaned and conformed data on S3';

-- Gold Schema (S3 External Storage)
CREATE SCHEMA IF NOT EXISTS workspace.`workspace-adventureworks-gold`
LOCATION 's3://your-bucket-name/adventureworks/gold'
COMMENT 'Gold layer - Business aggregates and metrics on S3';
```

**Required AWS Permissions**:
- S3 bucket access (s3:PutObject, s3:GetObject, s3:DeleteObject, s3:ListBucket)
- Instance profile or IAM role attached to Databricks workspace
- Unity Catalog external location configured

---

### Run Manually
1. Click **Run All** to execute the entire pipeline
2. Monitor progress in real-time with detailed logging
3. Review the final summary for execution results

**Note**: Serverless compute will auto-start for each notebook. First execution may include cold start time (~1-2 min per notebook).

---

### Schedule as Databricks Job

#### Option 1: Daily Full Refresh (Recommended)
```python
# Schedule: Every day at 2:00 AM
# Compute: Serverless SQL Warehouse or Serverless Compute
# Notebook: 05_Orchestration_Workflow
# Timeout: 90 minutes (includes cold starts + S3 writes)
# Retry: 1 retry on failure
```

#### Option 2: Hourly Incremental (Gold Layer Only)
```python
# Schedule: Every hour during business hours
# Only run Gold layer notebooks for near real-time metrics
# Comment out Bronze and Silver steps (cells 5, 6, 7)
# Timeout: 30 minutes
```

---

### Monitoring
- Check **Job Runs** in Databricks UI for execution history
- Set up **Email Alerts** on job failure
- Use **Databricks SQL Dashboards** to visualize Gold table metrics
- Monitor **Serverless Compute Usage** in the billing dashboard
- Monitor **S3 storage costs** in AWS Cost Explorer

---

### Troubleshooting

#### Common Issues
- **Cold start delays**: First run after idle period takes longer (1-2 min per notebook)
- **S3 access denied**: Verify IAM role has S3 bucket permissions
- **Schema not found**: Run the external schema creation SQL above
- **If Bronze fails**: Check source catalog connection and S3 write permissions
- **If Silver fails**: Verify Bronze tables exist in S3 and have data
- **If Gold fails**: Check Silver table quality and joins
- **Timeout errors**: Increase TIMEOUT_SECONDS (S3 I/O may be slower than managed storage)

#### S3-Specific Troubleshooting

**Error: "Access Denied" writing to S3**
```
Solution:
1. Verify instance profile attached to workspace
2. Check S3 bucket policy allows Databricks access
3. Verify Unity Catalog external location is configured
4. Test access: SELECT * FROM workspace.`workspace-adventureworks-bronze`.test_table
```

**Error: "Schema does not exist"**
```
Solution:
1. Create external schemas using SQL above
2. Replace 's3://your-bucket-name' with actual bucket
3. Ensure Unity Catalog external location exists
4. Verify: SHOW SCHEMAS IN workspace;
```

**Slow Performance**
```
Possible causes:
1. Network latency to S3 (especially cross-region)
2. Small file problem (too many small files)
3. Solution: Use OPTIMIZE command on Delta tables
4. Solution: Enable Photon acceleration (already enabled on Serverless)
```

---

### Performance Tuning for Serverless + S3

- **Sequential vs Parallel**: 
  - Current config: Sequential (MAX_WORKERS=1) for single-node serverless
  - To enable parallel: Set MAX_WORKERS=2 or 3 if serverless scales automatically
  - Parallel execution may reduce total time but increase cost

- **Cold Starts**: 
  - Schedule jobs to run consecutively to keep compute warm
  - Consider keeping Gold layer on hourly schedule to maintain warm sessions

- **S3 Performance**:
  - Delta Lake automatically optimizes S3 writes
  - Use OPTIMIZE command periodically to compact small files
  - Enable Z-ordering on frequently filtered columns
  - Photon acceleration improves S3 I/O performance

- **Cost Optimization**:
  - Serverless charges per-second for active compute
  - S3 storage: Standard tier for Bronze, Intelligent-Tiering for Silver/Gold
  - Sequential execution = predictable, lower concurrent usage
  - Estimated cost: $0.50-$1.00 per full pipeline run (compute) + S3 storage

---

### Benefits of S3 External Storage

- ✅ **Data Durability**: 99.999999999% (11 9's) durability
- ✅ **Cost-Effective**: Cheaper than managed storage for large datasets
- ✅ **Cross-Workspace Access**: Access same data from multiple Databricks workspaces
- ✅ **Data Portability**: Use data with other tools (Spark, Presto, Athena)
- ✅ **Lifecycle Policies**: Automatically archive old data to Glacier
- ✅ **Disaster Recovery**: S3 versioning and cross-region replication
- ✅ **Compliance**: Meets regulatory requirements for data retention

---

### Serverless Compute Benefits
- ✅ **Auto-scaling**: Adjusts resources based on workload
- ✅ **No cluster management**: Zero configuration required
- ✅ **Pay-per-use**: Charges only for active execution time
- ✅ **Fast startup**: 30-120 seconds cold start
- ✅ **Cost-effective**: Ideal for scheduled batch jobs
- ✅ **Photon enabled**: Optimized S3 I/O performance

---

## ✅ Pipeline Complete!

Your AdventureWorks analytics platform is now operational with:
- 🥉 **70+ Bronze tables** on S3 with audit trails
- 🥈 **5 Silver tables** on S3 with clean, conformed data
- 🥇 **3 Gold tables** on S3 ready for dashboards and reporting

**Optimized for**: 
- Databricks Serverless Compute (Single Node, Sequential Execution)
- AWS S3 External Storage (Unity Catalog External Schemas)

**Next Steps**:
1. Create Databricks SQL dashboards on Gold tables
2. Set up S3 lifecycle policies for cost optimization
3. Enable S3 versioning for data recovery
4. Monitor S3 costs in AWS Cost Explorer